In [ ]:
import pandas as pd

raw_detailed_df = pd.read_csv("./NACC_ADSP_PHC_Amyloid_Detailed_2024_merged.csv")
raw_append_df = pd.read_csv("./NACC_ADSP_PHC_demographic_2024.csv")
raw_df = pd.merge(raw_detailed_df, raw_append_df, on=["NACCID", "PHC_Age_PET"], how="outer")
raw_df = raw_df.loc[:, ~raw_df.columns.str.endswith("_y")]
raw_df = raw_df.rename(columns=lambda x: x[:-2] if x.endswith('_x') else x)
mapping_df = pd.read_csv("./column_mapping.csv")
pacctmci_df = pd.read_csv("./NACC_PACCTMCI.csv")

print(f"Initial data shape: {raw_df.shape}")

In [ ]:
filters = {
    "RACE": [1],
    "HISPANIC": [0],
    "PHC_TRACER": ["FBP"]
}


# ---- Apply row filters ----
filtered_df = raw_df.copy()
for col, allowed_values in filters.items():
    if col in filtered_df.columns:
        filtered_df = filtered_df[filtered_df[col].isin(allowed_values)]
        
print(f"Data shape after filtering: {filtered_df.shape}")

In [ ]:
# ---- Column mapping ----
mapping_df = mapping_df[mapping_df["TargetName"].notna() & (mapping_df["TargetName"] != "")]
column_mapping = dict(zip(mapping_df["SourceName"], mapping_df["TargetName"]))

# Ensure all expected columns exist in filtered_df
for src_col in column_mapping.keys():
    if src_col not in filtered_df.columns:
        filtered_df[src_col] = None  # or np.nan if preferred

# Apply mapping
filtered_df = filtered_df[list(column_mapping.keys())].rename(columns=column_mapping)

print(f"Data shape after column mapping: {filtered_df.shape}")

In [ ]:
value_mappings = {
    "DXGrp": {
        88.0: 1,    # CN
        1.0: 4,     # AD
    },
}

# ---- Apply value mappings ----
filtered_df["DXGrp"] = filtered_df["DXGrp"].map(value_mappings["DXGrp"])

pacctmci_lookup = pacctmci_df.rename(
    columns={
        "NACCID": "RID",
        "PHC_Age_PET": "AGE",
        "NACCTMCI": "PACCTMCI",
    }
)[["RID", "AGE", "PACCTMCI"]]

filtered_df = filtered_df.merge(pacctmci_lookup, on=["RID", "AGE"], how="left")

dxgrp_2_mask = filtered_df["DXGrp"].isna()
filtered_df.loc[
    dxgrp_2_mask & filtered_df["PACCTMCI"].isin([1, 2]),
    "DXGrp",
] = 2
filtered_df.loc[
    dxgrp_2_mask & ~filtered_df["PACCTMCI"].isin([1, 2]),
    "DXGrp",
] = -1

filtered_df["DXGrp"] = filtered_df["DXGrp"].astype("Int64")
filtered_df = filtered_df.drop(columns=["PACCTMCI"])

In [ ]:
# ---- Merge cognitive scores ----
raw_cogn_df = pd.read_csv("./NACC_ADSP_PHC_Cognition_2024_with_dates.csv")

# Step 1: Sort both dataframes to speed up lookups
filtered_df = filtered_df.sort_values(['RID', 'EXAMDATE'])
raw_cogn_df = raw_cogn_df.sort_values(['NACCID', 'VISIT_DATE'])

# Ensure date columns are datetime objects
filtered_df['EXAMDATE'] = pd.to_datetime(filtered_df['EXAMDATE'])
raw_cogn_df['VISIT_DATE'] = pd.to_datetime(raw_cogn_df['VISIT_DATE'])

# Step 2: Define a helper to find the nearest date match for each RID
def match_nearest_date(row):
    rid = row['RID']
    exam_date = row['EXAMDATE']
    
    subset = raw_cogn_df[raw_cogn_df['NACCID'] == rid]
    if subset.empty:
        return pd.Series({'PHC_MEM': None, 'PHC_EXF': None, 'PHC_LAN': None})
    
    # Find index of closest VISIT_DATE to EXAMDATE
    idx = (subset['VISIT_DATE'] - exam_date).abs().idxmin()
    matched = subset.loc[idx]
    return pd.Series({
        'PHC_MEM': matched['PHC_MEM'],
        'PHC_EXF': matched['PHC_EXF'],
        'PHC_LAN': matched['PHC_LAN']
    })

# Step 3: Apply to all rows
filtered_df[['PHC_MEM', 'PHC_EXF', 'PHC_LAN']] = filtered_df.apply(match_nearest_date, axis=1)

In [ ]:
# ---- Remove data points with missing values ----
# Identify the CTX_ columns
ctx_cols = [col for col in filtered_df.columns if col.startswith("CTX_")]

# Drop rows with NaN in those columns
filtered_df = filtered_df.dropna(subset=ctx_cols)

print(f"Final dataset shape: {filtered_df.shape}")

In [ ]:
# ---- split into single_visit and multi_visit ----
visit_counts = filtered_df['RID'].value_counts()
single_visit_rids = visit_counts[visit_counts == 1].index
multi_visit_rids = visit_counts[visit_counts > 1].index
single_visit_df = filtered_df[filtered_df['RID'].isin(single_visit_rids)]
multi_visit_df = filtered_df[filtered_df['RID'].isin(multi_visit_rids)]

print(f"Single-visit data shape: {single_visit_df.shape}")
print(f"Multi-visit data shape: {multi_visit_df.shape}")

In [ ]:
# Save output
single_visit_df.to_csv("../filtered/single_visit_data.csv", index=False)
multi_visit_df.to_csv("../filtered/multi_visit_data.csv", index=False)